<a href="https://colab.research.google.com/github/zakateryila/Zaka-Teryila-Peter-/blob/main/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
!pip install -q importlib-resources

import tensorflow_datasets as tfds
ds, info = tfds.load("oxford_iiit_pet", split="train", with_info=True)
print(info.features["species"].names) # cat / dog
print(info.features["label"].names) # 37 breed names

['Cat', 'Dog']
['Abyssinian', 'american_bulldog', 'american_pit_bull_terrier', 'basset_hound', 'beagle', 'Bengal', 'Birman', 'Bombay', 'boxer', 'British_Shorthair', 'chihuahua', 'Egyptian_Mau', 'english_cocker_spaniel', 'english_setter', 'german_shorthaired', 'great_pyrenees', 'havanese', 'japanese_chin', 'keeshond', 'leonberger', 'Maine_Coon', 'miniature_pinscher', 'newfoundland', 'Persian', 'pomeranian', 'pug', 'Ragdoll', 'Russian_Blue', 'saint_bernard', 'samoyed', 'scottish_terrier', 'shiba_inu', 'Siamese', 'Sphynx', 'staffordshire_bull_terrier', 'wheaten_terrier', 'yorkshire_terrier']


In [20]:
ds, info = tfds.load("caltech101", split="train", with_info=True)
print(info.features["label"].names) # print this and read it properly

['accordion', 'airplanes', 'anchor', 'ant', 'background_google', 'barrel', 'bass', 'beaver', 'binocular', 'bonsai', 'brain', 'brontosaurus', 'buddha', 'butterfly', 'camera', 'cannon', 'car_side', 'ceiling_fan', 'cellphone', 'chair', 'chandelier', 'cougar_body', 'cougar_face', 'crab', 'crayfish', 'crocodile', 'crocodile_head', 'cup', 'dalmatian', 'dollar_bill', 'dolphin', 'dragonfly', 'electric_guitar', 'elephant', 'emu', 'euphonium', 'ewer', 'faces', 'faces_easy', 'ferry', 'flamingo', 'flamingo_head', 'garfield', 'gerenuk', 'gramophone', 'grand_piano', 'hawksbill', 'headphone', 'hedgehog', 'helicopter', 'ibis', 'inline_skate', 'joshua_tree', 'kangaroo', 'ketch', 'lamp', 'laptop', 'leopards', 'llama', 'lobster', 'lotus', 'mandolin', 'mayfly', 'menorah', 'metronome', 'minaret', 'motorbikes', 'nautilus', 'octopus', 'okapi', 'pagoda', 'panda', 'pigeon', 'pizza', 'platypus', 'pyramid', 'revolver', 'rhino', 'rooster', 'saxophone', 'schooner', 'scissors', 'scorpion', 'sea_horse', 'snoopy', 's

In [21]:
ds = tfds.load("stanford_dogs", split="train")

In [22]:
import tensorflow as tf
import tensorflow_datasets as tfds

# Define target image size
IMG_SIZE = (224, 224)

def preprocess_img(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

# 1. Cats and Dogs from Oxford IIIT Pet Dataset
pet_ds, pet_info = tfds.load('oxford_iiit_pet', split='train', with_info=True, as_supervised=False)

def map_pet(feat):
    # species: 0 is cat, 1 is dog
    # We map Cat to class 0, Dog to class 1
    return feat['image'], tf.cast(feat['species'], tf.int32)

pet_mapped = pet_ds.map(map_pet).map(preprocess_img)

# 2. Neither category from Caltech101 (excluding cats/dogs if any, keeping it general)
caltech_ds, caltech_info = tfds.load('caltech101', split='train', with_info=True, as_supervised=True)

# Map Caltech101 objects to class 2 (Neither)
def map_neither(image, label):
    return image, tf.constant(2, dtype=tf.int32)

neither_mapped = caltech_ds.map(map_neither).map(preprocess_img)

# Take subsets to balance the classes somewhat
cat_dog_samples = pet_mapped.take(3000)
neither_samples = neither_mapped.take(1500)

# Combine datasets
combined_dataset = cat_dog_samples.concatenate(neither_samples).shuffle(buffer_size=4000)

# Batch and prefetch
BATCH_SIZE = 32
train_dataset = combined_dataset.batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)

print("Combined dataset pipeline created successfully.")

Combined dataset pipeline created successfully.


In [15]:
from tensorflow.keras import layers, models

# 1. Load pretrained MobileNetV2 base
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Freeze base layers

# 2. Build the model architecture
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(3, activation='softmax')  # 3 Output classes (Cat, Dog, Neither)
])

# 3. Compile the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# 4. Train the model
print("Starting training...")
history = model.fit(
    train_dataset,
    epochs=5
)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,339 (9.24 MB)

 Trainable params: 164,355 (642.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Starting training...
Epoch 1/5
141/141 ━━━━━━━━━━━━━━━━━━━━ 37s 67ms/step - accuracy: 0.9591 - loss: 0.1236
Epoch 2/5
141/141 ━━━━━━━━━━━━━━━━━━━━ 28s 36ms/step - accuracy: 0.9878 - loss: 0.0335
Epoch 3/5
141/141 ━━━━━━━━━━━━━━━━━━━━ 14s 35ms/step - accuracy: 0.9949 - loss: 0.0148
Epoch 4/5
141/141 ━━━━━━━━━━━━━━━━━━━━ 13s 35ms/step - accuracy: 0.9976 - loss: 0.0065
Epoch 5/5
141/141 ━━━━━━━━━━━━━━━━━━━━ 15s 35ms/step - accuracy: 0.9993 - loss: 0.0049


In [16]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing import image

def classify_images_in_folder(folder_path, model):
    class_names = {0: 'cat', 1: 'dog', 2: 'neither'}
    results = []

    # Ensure directory exists
    if not os.path.exists(folder_path):
        print(f"Directory '{folder_path}' does not exist. Please create it and upload your images.")
        return pd.DataFrame()

    # Iterate through all files in the folder
    for filename in sorted(os.listdir(folder_path)):
        file_path = os.path.join(folder_path, filename)

        # Process only images
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.bmp')):
            try:
                # Load, resize, and convert to array
                img = image.load_img(file_path, target_size=(224, 224))
                img_array = image.img_to_array(img)
                # Normalize input matching the training prep
                img_array = img_array / 255.0
                img_array = np.expand_dims(img_array, axis=0)

                # Make prediction
                predictions = model.predict(img_array, verbose=0)[0]
                predicted_class_id = np.argmax(predictions)
                confidence = predictions[predicted_class_id] * 100

                results.append({
                    'filename': filename,
                    'prediction': class_names[predicted_class_id],
                    'confidence': f"{confidence:.1f}%"
                })
            except Exception as e:
                print(f"Could not process image {filename}: {e}")

    # Create a DataFrame to display nicely
    df_results = pd.DataFrame(results)
    return df_results

# Helper placeholder: Create a dummy test folder so the script runs successfully immediately
test_dir = '/content/test_photos'
os.makedirs(test_dir, exist_ok=True)
print(f"Please upload your evaluation photographs to: {test_dir}")

Please upload your evaluation photographs to: /content/test_photos


In [17]:
# Once you have uploaded images to '/content/test_photos', execute this cell to get the predictions:
predictions_df = classify_images_in_folder('/content/test_photos', model)
if not predictions_df.empty:
    display(predictions_df)

### 🌟 Interactive Cat, Dog, or Neither Predictor 🌟
Use the interactive uploader below to quickly test your trained model with any photo!

In [18]:
!pip install -q gradio

import gradio as gr
import numpy as np
from tensorflow.keras.preprocessing import image as keras_image

def predict_image(input_image):
    if input_image is None:
        return "No image uploaded."

    # Resize and preprocess to match model requirements
    img_resized = input_image.resize((224, 224))
    img_array = keras_image.img_to_array(img_resized)
    img_array = img_array / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # Run model prediction
    predictions = model.predict(img_array, verbose=0)[0]
    class_names = {0: 'Cat 🐱', 1: 'Dog 🐶', 2: 'Neither ❓'}

    # Return a dictionary of labels and their probabilities for Gradio's Label component
    return {class_names[i]: float(predictions[i]) for i in range(3)}

# Design custom Gradio Theme & Interface
demo = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="pil", label="Upload an Image"),
    outputs=gr.Label(num_top_classes=3, label="Prediction probabilities"),
    title="🐱 Cat, Dog, or Neither Classifier 🐶",
    description="Upload any photograph to identify if it features a cat, a dog, or neither of them! Powered by MobileNetV2.",
    theme="soft"
)

# Launch with a shareable public URL
demo.launch(share=True)

/usr/local/lib/python3.13/dist-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a4d03d335399b5e49d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
